# BERTopic — Template para Novo Corpus

Copie este notebook para `notebooks/bertopic/01_bertopic_<corpus>.ipynb` e
substitua os placeholders marcados com `# <<` para adaptar a um novo corpus.

## Pipeline (5 etapas)
1. **Pré-processamento** — stop words customizadas, CountVectorizer com acentos
2. **Embeddings** — carregados do cache do `02-embeddings` (**nunca** computados aqui)
3. **Redução + Clustering** — UMAP 5D → HDBSCAN (leaf)
4. **Representação** — c-TF-IDF + MMR (diversity=0.2)
5. **Pós-processamento** — reduce_outliers → update_topics → reduce_topics → update_topics → nomeação LLM

## 6 Métricas avaliadas
Topic Diversity, Taxa de outliers, Coerência C_v, Coerência Semântica,
Exclusividade, FREX (Frequency × Exclusivity)

## Pré-requisito: rodar o `02-embeddings` primeiro

Os embeddings são responsabilidade **exclusiva** do módulo 02 — ele trata o
contexto do Ollama (`params.yaml > embeddings.models[].max_tokens` → `num_ctx`),
crítico para documentos longos. Este notebook apenas **lê** o `.npy` versionado
mais recente de `02-embeddings/data/output/<corpus>/<stamp>/`; se não existir,
falha com instrução explícita em vez de computar inline.

## Saídas

No **run dir** carimbado (`data/output/<corpus>/bertopic/<corpus>_<stamp>/`):

- `bertopic_results.csv`, `bertopic_topics_for_eval.csv`, `bertopic_metrics.csv`
- `bertopic_docs_por_topico.png`
- `bertopic_evolucao_temporal.png`, `bertopic_heatmap_temporal.png`,
  `bertopic_evolucao_por_topico.png` (se o corpus declarar `date_column`)

> Os nomes acima são os que o site Quarto referencia — mantenha-os ao adaptar.

## Antes de executar via `nbconvert` — registre o kernel

O `kernelspec` gravado no `.ipynb` é `python3`, que pode resolver para um Python
diferente do `venv` do projeto (Python do sistema, ou venv de outro projeto) e
falhar com `ModuleNotFoundError` mesmo com tudo instalado. Registre o venv uma vez
por máquina e aponte o `nbconvert` para ele:

```powershell
venv\Scripts\python -m ipykernel install --user --name topic-modeling --display-name "Topic Modeling (venv)"

venv\Scripts\python -m jupyter nbconvert --to notebook --execute --inplace `
  "03-topic-modeling/notebooks/bertopic/01_bertopic_<corpus>.ipynb" `
  --ExecutePreprocessor.kernel_name=topic-modeling --ExecutePreprocessor.timeout=21600
```

In [ ]:
# Descomente se necessário
# !pip install bertopic gensim sentence-transformers umap-learn hdbscan httpx nest_asyncio itables wordcloud -q
# !python -m spacy download pt_core_news_lg -q

In [ ]:
# ── path fix: resolve _helpers.py e data/ de qualquer subdiretório ──
import sys as _sys, os as _os
from pathlib import Path as _P
_here = _P().resolve()
_nb_dir = _here if (_here / '_helpers.py').exists() else _here.parent
if str(_nb_dir) not in _sys.path:
    _sys.path.insert(0, str(_nb_dir))
_os.chdir(_nb_dir)          # '../data/...' paths ficam corretos
del _here, _nb_dir, _P, _sys, _os
# ─────────────────────────────────────────────────────────────────────
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from sklearn.feature_extraction import text as sk_text
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

from _helpers import (
    load_params, get_corpus_config, get_column_names, get_seed, load_corpus,
    resolve_latest_dir, make_run_output_dir, name_all_topics,
    compute_coherence_cv, compute_embedding_coherence, compute_topic_diversity,
    compute_exclusivity_ctfidf, compute_frex_score,
    export_results, export_topics_for_eval,
)

# ── CONFIGURAÇÃO — edite aqui para cada corpus ──────────────────────────
CORPUS = "social"              # << nome do corpus no params.yaml
# ────────────────────────────────────────────────────────────────────────

params = load_params()
corpus_id, corpus_cfg = get_corpus_config(params, CORPUS)
cols = get_column_names(corpus_cfg)
seed = get_seed(params)
np.random.seed(seed)

TEXT_COL    = cols["text"]
POST_ID_COL = cols["post_id"]
DATE_COL    = corpus_cfg.get("date_column")          # opcional — habilita as figuras temporais
LANGUAGE    = corpus_cfg.get("language", "pt")       # "pt" ou "en"
BT_LANGUAGE = corpus_cfg.get("bertopic_language", "multilingual")

bt_cfg = params["bertopic"]
TOKEN_PATTERN = corpus_cfg.get(
    "token_pattern", bt_cfg["vectorizer"].get("token_pattern", r"(?u)\w{2,}")
)

subdir     = corpus_cfg.get("subdir", corpus_id)
INPUT_DIR  = Path(f"../data/input/{subdir}")         # fallback legado (latest-wins e o padrao)
BASE_OUTPUT_DIR = Path(f"../data/output/{subdir}/bertopic")
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Saida carimbada por execucao: <corpus>_<data>_<hora> (nao sobrescreve runs anteriores)
OUTPUT_DIR = make_run_output_dir(BASE_OUTPUT_DIR, corpus_id)

MODEL_NAME = bt_cfg["embedding_model"]
BACKEND    = bt_cfg.get("embedding_backend", "ollama")
DIMENSION  = bt_cfg.get("embedding_dimension")       # None → dimensao nativa do modelo
SAFE_MODEL = re.sub(r'[/:\?*"<>|]', "_", MODEL_NAME)  # Windows-safe
SUFFIX     = bt_cfg.get("embedding_suffix", "4096d")
EMB_FILENAME = f"embeddings_{SAFE_MODEL}_{SUFFIX}.npy"

print(f"Corpus   : {corpus_id}  |  seed={seed}  |  lang={LANGUAGE}")
print(f"Modelo   : {MODEL_NAME}  |  Dim: {DIMENSION or 'nativo'}  |  Backend: {BACKEND}")
print(f"Run dir  : {OUTPUT_DIR}")

In [ ]:
# Corpus: resolve o output mais recente do 01-preprocessing (latest-wins, sem copia)
CORPUS_DIR = resolve_latest_dir(
    f"../../01-preprocessing/data/output/{subdir}",
    contains="corpus_limpo.csv",
    fallback_dir=str(INPUT_DIR),
)
CORPUS_VERSION = CORPUS_DIR.name
print(f"Versao do corpus: {CORPUS_VERSION}")

df = load_corpus(CORPUS_DIR)
if POST_ID_COL not in df.columns:
    df[POST_ID_COL] = [f"doc_{i:04d}" for i in range(len(df))]
docs     = df[TEXT_COL].astype(str).tolist()
post_ids = df[POST_ID_COL].astype(str).tolist()

# Embeddings: SO leitura de cache. O 03 NAO computa embeddings inline — essa
# responsabilidade e exclusiva do 02-embeddings (que trata num_ctx/contexto para
# documentos longos via params.yaml > embeddings.models[].max_tokens).
_emb_base = Path("../../02-embeddings/data/output") / subdir
try:
    _emb_dir = resolve_latest_dir(_emb_base, contains=EMB_FILENAME, fallback_dir=str(_emb_base))
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"Embeddings nao encontrados em {_emb_base}. Rode "
        "02-embeddings/notebooks/02_embeddings.ipynb para este corpus antes do "
        "BERTopic (o 03 nao computa embeddings inline)."
    ) from exc
EMBEDDINGS_VERSION = _emb_dir.name
CACHE_PATH = _emb_dir / EMB_FILENAME
assert CACHE_PATH.exists(), f"Embeddings nao encontrados em {CACHE_PATH}"

embeddings = np.load(CACHE_PATH)
assert embeddings.shape[0] == len(docs), (
    f"Desalinhamento: embeddings={embeddings.shape[0]} vs docs={len(docs)}. "
    "Regenerar embeddings via 02-embeddings antes de prosseguir."
)

print(f"Versao dos embeddings: {EMBEDDINGS_VERSION}")
print(f"Docs       : {len(docs)}")
print(f"Embeddings : {embeddings.shape}")
print(f"\nExemplo: {docs[0][:120]}")

## Pré-processamento — CountVectorizer

O CountVectorizer do BERTopic gera as keywords (c-TF-IDF) — **não** os embeddings.
Os três ajustes críticos:

1. **Stop words customizadas** — NLTK base + termos de domínio específicos do corpus
2. **ngram_range=(1,2)** — habilita bigramas relevantes (ex: "saúde pública")
3. **token_pattern** — preserva acentos PT-BR (`À-ÿ`) e tokens de emojis demojizados (`_`)

In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download("stopwords", quiet=True)

# Stop words base por idioma
if LANGUAGE == "pt":
    base_sw = set(stopwords.words("portuguese"))
else:
    base_sw = set(sk_text.ENGLISH_STOP_WORDS)

# << Adicione stop words de domínio do corpus abaixo
STOPWORDS_DOMAIN = set([
    # Ex para social: "raquellyra", "governodepernambuco", "pernambuco"
])

# Stop words de emojis demojizados — vêm de params.yaml > corpora.<corpus>.stopwords_emojis
STOPWORDS_EMOJIS = set(corpus_cfg.get("stopwords_emojis", []))

all_stop = base_sw | STOPWORDS_DOMAIN | STOPWORDS_EMOJIS

vectorizer = CountVectorizer(
    stop_words=list(all_stop),
    ngram_range=tuple(bt_cfg["vectorizer"]["ngram_range"]),
    token_pattern=TOKEN_PATTERN,
    min_df=2,
)
print(f"Stop words total: {len(all_stop)}")

In [ ]:
umap_p = bt_cfg["umap"]
umap_model = UMAP(
    n_components=umap_p["n_components"],
    n_neighbors=umap_p["n_neighbors"],
    min_dist=umap_p["min_dist"],
    metric=umap_p["metric"],
    random_state=seed,
)

hdbscan_p = bt_cfg["hdbscan"]
hdbscan_model = HDBSCAN(
    min_cluster_size=hdbscan_p["min_cluster_size"],
    min_samples=hdbscan_p["min_samples"],
    cluster_selection_method=hdbscan_p["cluster_selection_method"],
    prediction_data=True,
)

# MMR como ASPECTO, nao a representacao Main: get_topic(tid) (sem full=True)
# continua devolvendo o c-TF-IDF puro -- o que toda metrica de qualidade deve
# consumir; a lista diversificada fica em get_topic(tid, full=True)["MMR"] para
# quem precisa dela -- nomeacao via LLM e leitura humana.
representation_model = {
    "MMR": MaximalMarginalRelevance(
        diversity=bt_cfg["mmr_diversity"],
        top_n_words=params["evaluation"].get("top_n_words_cache", 100),
    ),
}

topic_model = BERTopic(
    language=BT_LANGUAGE,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    representation_model=representation_model,
    min_topic_size=bt_cfg["min_topic_size"],
    calculate_probabilities=True,
    verbose=True,
)

t0 = time.time()
topics, probs = topic_model.fit_transform(docs, embeddings)
print(f"\nTreinamento: {time.time()-t0:.1f}s")
print(f"Tópicos encontrados: {topic_model.get_topic_info().shape[0] - 1} (+1 outlier)")
print(f"Outliers (-1): {sum(t == -1 for t in topics)} ({sum(t == -1 for t in topics)/len(topics):.1%})")

In [ ]:
# ORDEM CORRETA do pos-processamento (4 etapas):
#   reduce_outliers  → retorna novos assignments (NÃO atualiza o modelo)
#   update_topics    → sincroniza o modelo com os assignments sem outliers  ← obrigatório antes de reduce_topics
#   reduce_topics    → mescla os menores tópicos até nr_topics
#   update_topics    → atualiza representações c-TF-IDF + MMR após a redução
# Pular o 1º update_topics faz reduce_topics operar sobre assignments desatualizados.

# 1. Reatribuir outliers via c-TF-IDF
new_topics = topic_model.reduce_outliers(docs, topics, strategy="c-tf-idf")

# 2. Sincronizar modelo com assignments sem outliers
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer,
                           representation_model=representation_model)

# 3. Reduzir para K tópicos alvo
topic_model.reduce_topics(docs, nr_topics=bt_cfg["reduce_topics_nr"])
new_topics = topic_model.topics_

# 4. Atualizar representações c-TF-IDF + MMR com K final
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer,
                           representation_model=representation_model)
topics = new_topics
topic_info = topic_model.get_topic_info()

print(f"Tópicos após redução: {topic_info[topic_info.Topic != -1].shape[0]}")
print(f"Outliers remanescentes: {sum(t == -1 for t in topics)}")
topic_info.head(10)

In [ ]:
llm_cfg = params.get("llm", {})
valid_ids = [t for t in topic_info["Topic"].tolist() if t != -1]
top_n_llm = params["evaluation"].get("top_n_keywords_llm", 10)

# MMR (diversificado): so para nomeacao via LLM / leitura humana. As metricas
# de qualidade (proxima celula, topics_keywords_metrics) vem do c-TF-IDF puro
# (topic_model.get_topic sem full=True) -- nunca da lista MMR.
topics_keywords = {
    tid: [w for w, _ in topic_model.get_topic(tid, full=True)["MMR"][:top_n_llm]]
    for tid in valid_ids
}
example_docs_map = {tid: topic_model.get_representative_docs(tid) for tid in valid_ids}

print(f"Nomeando {len(valid_ids)} tópicos via {llm_cfg.get('model')} (lang={LANGUAGE})...")
t0 = time.time()
try:
    rotulos = name_all_topics(
        topics_keywords,
        model=llm_cfg["model"],
        base_url=llm_cfg.get("base_url", "http://localhost:11434"),
        example_docs_map=example_docs_map,
        lang=LANGUAGE,
    )
    print(f"done em {time.time()-t0:.0f}s")
except Exception as e:
    print(f"LLM falhou ({e}) — fallback top-3 keywords")
    rotulos = {tid: ", ".join(kws[:3]) for tid, kws in topics_keywords.items()}

for tid in sorted(rotulos):
    print(f"  T{tid}: {rotulos[tid]}")

topic_model.set_topic_labels(rotulos)

In [ ]:
top_n_metrics = params["evaluation"].get("top_n_keywords_metrics", 20)
topics_keywords_metrics = {
    tid: [w for w, _ in topic_model.get_topic(tid)[:top_n_metrics]] for tid in valid_ids
}

# 1. Taxa de outliers
outlier_rate = sum(t == -1 for t in topics) / len(topics)

# 2. Topic Diversity (Dieng et al., 2020)
td_score = compute_topic_diversity(topics_keywords_metrics, top_k=top_n_metrics)

# 3. Coerência C_v via gensim (tokenização lematizada, idioma-aware)
#    Amostra de até 5000 docs (seed 42) — PADRONIZADO com os notebooks de corpus
#    (evita lematização lenta de spaCy em corpora grandes).
from _helpers import lemmatize_corpus
_sample_idx = np.random.RandomState(42).choice(len(docs), min(5000, len(docs)), replace=False)
_docs_sample = [docs[i] for i in _sample_idx]
tokenized, dictionary = lemmatize_corpus(_docs_sample, LANGUAGE, params)
cv_score = compute_coherence_cv(topics_keywords_metrics, tokenized, dictionary)

# 4. Coerência Semântica (similaridade de embeddings entre keywords)
if BACKEND == "ollama":
    from _helpers import get_ollama_embeddings
    def _embed(texts):
        return get_ollama_embeddings(list(texts), MODEL_NAME, dimension=DIMENSION)
else:
    from sentence_transformers import SentenceTransformer
    _st_model = SentenceTransformer(MODEL_NAME)
    def _embed(texts):
        return _st_model.encode(list(texts), show_progress_bar=False, convert_to_numpy=True)

sem_score = compute_embedding_coherence(topics_keywords_metrics, _embed)

# 5. Exclusividade — c-TF-IDF completo (não só top-N do próprio tópico)
ctfidf_full = topic_model.c_tf_idf_.toarray()
vocab_full  = topic_model.vectorizer_model.get_feature_names_out()
topic_idx   = {tid: i for i, tid in enumerate(sorted(topic_model.get_topics().keys()))}
topic_word_scores = {
    tid: {vocab_full[j]: float(ctfidf_full[topic_idx[tid]][j])
          for j in range(len(vocab_full)) if ctfidf_full[topic_idx[tid]][j] > 0}
    for tid in valid_ids
}
excl_score, excl_per_topic = compute_exclusivity_ctfidf(
    topics_keywords_metrics, topic_word_scores, top_n=top_n_metrics
)

# 6. FREX (Airoldi & Bischof, 2016 — harmônica freq × exclusividade)
vocab_index = {w: i for i, w in enumerate(vocab_full)}
frex_score, frex_per_topic = compute_frex_score(
    topics_keywords_metrics, topic_word_matrix=ctfidf_full,
    vocab_index=vocab_index, topic_index=topic_idx, top_n=top_n_metrics, w_freq=0.5,
)

print("=" * 55)
print(f"  Tópicos válidos      : {len(valid_ids)}")
print(f"  Taxa de outliers     : {outlier_rate:.1%}")
print(f"  Topic Diversity      : {td_score:.4f}  (↑ melhor)")
print(f"  Coerência C_v        : {cv_score:.4f}              (↑ melhor)")
print(f"  Coer. Semântica      : {sem_score:.4f}              (↑ melhor)")
print(f"  Exclusividade        : {excl_score:.4f}              (↑ melhor)")
print(f"  FREX                 : {frex_score:.4f}              (↑ melhor)")
print("=" * 55)

## Sweep de estrategias/threshold de reduce_outliers + grid UMAP/HDBSCAN

Usa `_helpers.sweep_outlier_strategies` / `sweep_outlier_threshold` /
`sweep_bertopic_grid` para comparar empiricamente, via multiplos seeds,
alternativas ao pipeline fixo acima (pos-processamento em 4 etapas com a estrategia/nn/mcs do
`params.yaml`). Metricas: C_v, Topic Diversity, Exclusividade c-TF-IDF, FREX
e estabilidade Jaccard multi-seed.

> **Custoso** (refit completo por combinacao x seed). Habilitar via
> `evaluation.run_outlier_sweep` / `evaluation.run_param_sweep` no
> `params.yaml`. Por padrao ambos sao pulados.


In [ ]:
eval_cfg = params["evaluation"]
top_n_sweep_metrics = eval_cfg.get("top_n_keywords_metrics", 20)
from _helpers import sweep_outlier_strategies, sweep_outlier_threshold, sweep_bertopic_grid


def _sweep_vectorizer():
    """Vectorizer FRESCO por modelo do sweep — update_topics re-fita o
    vectorizer in-place; reusar a instancia global `vectorizer` corromperia
    seu vocab entre variantes."""
    return CountVectorizer(
        stop_words=list(all_stop),
        ngram_range=tuple(bt_cfg["vectorizer"]["ngram_range"]),
        token_pattern=TOKEN_PATTERN, min_df=2,
    )


def _sweep_build_model(seed, n_neighbors=None, min_cluster_size=None, min_samples=None, cluster_selection_method=None):
    nn = n_neighbors if n_neighbors is not None else umap_p["n_neighbors"]
    mcs = min_cluster_size if min_cluster_size is not None else hdbscan_p["min_cluster_size"]
    ms = min_samples if min_samples is not None else hdbscan_p["min_samples"]
    csm = cluster_selection_method if cluster_selection_method is not None else hdbscan_p.get("cluster_selection_method", "leaf")
    umap_s = UMAP(
        n_components=umap_p["n_components"], n_neighbors=nn,
        min_dist=umap_p["min_dist"], metric=umap_p["metric"],
        random_state=seed,
    )
    hdb_s = HDBSCAN(
        min_cluster_size=mcs, min_samples=ms,
        cluster_selection_method=csm,
        prediction_data=True,
    )
    return BERTopic(
        language=BT_LANGUAGE, umap_model=umap_s, hdbscan_model=hdb_s,
        vectorizer_model=_sweep_vectorizer(),
        representation_model={"MMR": MaximalMarginalRelevance(diversity=bt_cfg["mmr_diversity"], top_n_words=params["evaluation"].get("top_n_words_cache", 100))},
        min_topic_size=bt_cfg["min_topic_size"], calculate_probabilities=True, verbose=False,
    )


if eval_cfg.get("run_outlier_sweep", False):
    sweep_seeds = eval_cfg.get("outlier_sweep_seeds", [42, 123, 456, 789, 1024])
    sweep_strategies = eval_cfg.get(
        "outlier_sweep_strategies",
        ["off", "c-tf-idf", "embeddings", "probabilities", "distributions"],
    )
    print(f"Sweep de estrategias ({len(sweep_strategies)} estrategias x {len(sweep_seeds)} seeds)...")
    raw_strat, agg_strat = sweep_outlier_strategies(
        lambda s: _sweep_build_model(s), docs, embeddings, tokenized, dictionary,
        sweep_seeds, strategies=sweep_strategies,
        reduce_nr=bt_cfg.get("reduce_topics_nr"), top_n_metrics=top_n_sweep_metrics,
    )
    raw_strat.to_csv(OUTPUT_DIR / "sweep_outlier_strategies_raw.csv", index=False)
    agg_strat.to_csv(OUTPUT_DIR / "sweep_outlier_strategies.csv", index=False)
    print(agg_strat.to_string(index=False))
else:
    print("Sweep de estrategias desabilitado (evaluation.run_outlier_sweep=false).")
    agg_strat = pd.DataFrame()

if eval_cfg.get("run_threshold_sweep", False):
    thr_seeds = eval_cfg.get("threshold_sweep_seeds", [42])
    thr_grids = eval_cfg.get("threshold_sweep_grids", {})
    if thr_grids:
        n_pts = sum(len(v) for v in thr_grids.values())
        print(f"Sweep de threshold ({len(thr_grids)} estrategias x {n_pts} pontos x {len(thr_seeds)} seeds)...")
        raw_thr, agg_thr = sweep_outlier_threshold(
            lambda s: _sweep_build_model(s), docs, embeddings, tokenized, dictionary,
            thr_seeds, grids=thr_grids,
            reduce_nr=bt_cfg.get("reduce_topics_nr"), top_n_metrics=top_n_sweep_metrics,
        )
        raw_thr.to_csv(OUTPUT_DIR / "sweep_outlier_threshold_raw.csv", index=False)
        agg_thr.to_csv(OUTPUT_DIR / "sweep_outlier_threshold.csv", index=False)
        print(agg_thr.to_string(index=False))
    else:
        print("threshold_sweep_grids vazio — nada a varrer.")
else:
    print("Sweep de threshold desabilitado (evaluation.run_threshold_sweep=false).")
    agg_thr = pd.DataFrame()

if eval_cfg.get("run_param_sweep", False):
    grid_seeds = eval_cfg.get("param_sweep_seeds", [42, 123, 456])
    nn_grid = eval_cfg.get("param_sweep_n_neighbors", [umap_p["n_neighbors"]])
    mcs_grid = eval_cfg.get("param_sweep_min_cluster_size", [hdbscan_p["min_cluster_size"]])
    ms_grid = eval_cfg.get("param_sweep_min_samples", [None])  # [None] = usa o min_samples padrao do corpus
    csm_grid = eval_cfg.get("param_sweep_cluster_selection_methods", ["leaf"])
    print(f"Grid UMAP/HDBSCAN ({len(nn_grid)} nn x {len(mcs_grid)} mcs x {len(ms_grid)} ms x {len(csm_grid)} csm x {len(grid_seeds)} seeds)...")
    raw_grid, agg_grid = sweep_bertopic_grid(
        _sweep_build_model, docs, embeddings, tokenized, dictionary, grid_seeds,
        n_neighbors_grid=nn_grid, min_cluster_size_grid=mcs_grid, min_samples_grid=ms_grid,
        cluster_selection_methods_grid=csm_grid,
        reduce_nr_grid=[bt_cfg.get("reduce_topics_nr")],
        outlier_strategy="off",  # sempre off no grid: isola efeito estrutural
        top_n_metrics=top_n_sweep_metrics,
    )
    raw_grid.to_csv(OUTPUT_DIR / "sweep_bertopic_grid_raw.csv", index=False)
    agg_grid.to_csv(OUTPUT_DIR / "sweep_bertopic_grid.csv", index=False)
    print(agg_grid.to_string(index=False))
else:
    print("Grid UMAP/HDBSCAN desabilitado (evaluation.run_param_sweep=false).")
    agg_grid = pd.DataFrame()


In [ ]:
export_results(
    topics=topics, probs=probs, names=rotulos, texts=docs,
    output_path=str(OUTPUT_DIR / "bertopic_results.csv"),
    topic_type="semantic", granularity="unit", post_ids=post_ids,
)
export_topics_for_eval(
    topics_keywords=topics_keywords_metrics, topics_names=rotulos,
    model_name="bertopic", output_path=str(OUTPUT_DIR / "bertopic_topics_for_eval.csv"),
)

metrics_df = pd.DataFrame([{
    "corpus": corpus_id, "n_topics": len(valid_ids), "outlier_rate": outlier_rate,
    "topic_diversity": td_score, "coherence_cv": cv_score,
    "semantic_coherence": sem_score, "exclusivity": excl_score, "frex": frex_score,
}])
metrics_df.to_csv(OUTPUT_DIR / "bertopic_metrics.csv", index=False)

print(f"Salvo: {OUTPUT_DIR / 'bertopic_results.csv'}  ({len(docs)} docs)")
print(f"Salvo: {OUTPUT_DIR / 'bertopic_topics_for_eval.csv'}")
print(f"Salvo: {OUTPUT_DIR / 'bertopic_metrics.csv'}")

In [ ]:
# Distribuição de docs por tópico
counts = pd.Series(topics).value_counts().sort_index()
counts_valid = counts[counts.index != -1]

plt.figure(figsize=(12, 4))
plt.bar(counts_valid.index, counts_valid.values, color="steelblue")
plt.xlabel("Tópico")
plt.ylabel("Documentos")
plt.title(f"Distribuição de documentos por tópico — {corpus_id}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bertopic_docs_por_topico.png", dpi=150, bbox_inches="tight", facecolor="white")
print(f"Salvo: {OUTPUT_DIR / 'bertopic_docs_por_topico.png'}")
plt.show()

# Barchart interativo (requer plotly)
try:
    fig = topic_model.visualize_barchart(top_n_topics=min(12, len(valid_ids)), n_words=10)
    fig.show()
except Exception as e:
    print(f"visualize_barchart indisponível: {e}")

## Evolução temporal dos tópicos

Três leituras da mesma série mensal (contagem por tópico dominante), com os
mesmos nomes de arquivo dos notebooks de produção — o site Quarto depende deles.
A célula se pula sozinha se o corpus não declarar `date_column` no `params.yaml`.

In [ ]:
if DATE_COL and DATE_COL in df.columns:
    import seaborn as sns

    _tmp = pd.DataFrame({"date": pd.to_datetime(df[DATE_COL], errors="coerce"),
                        "topic_id": topics}).dropna(subset=["date"])
    _tmp = _tmp[_tmp["topic_id"] != -1]
    _tmp["mes"] = _tmp["date"].dt.to_period("M").astype(str)
    serie = _tmp.groupby(["mes", "topic_id"]).size().unstack(fill_value=0).sort_index()

    def _label(tid):
        return f"T{tid}: {str(rotulos.get(tid, ''))[:28]}"

    # V1 — linhas dos topicos mais volumosos
    top_ids = list(serie.sum().nlargest(min(8, serie.shape[1])).index)
    fig, ax = plt.subplots(figsize=(13, 5))
    for tid in top_ids:
        ax.plot(serie.index, serie[tid], marker="o", linewidth=1.8, markersize=4, label=_label(tid))
    ax.set_xlabel("Mes"); ax.set_ylabel("# documentos")
    ax.set_title(f"Evolucao temporal dos topicos (top {len(top_ids)}) — {corpus_id}")
    ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "bertopic_evolucao_temporal.png", dpi=150, bbox_inches="tight", facecolor="white")
    print(f"Salvo: {OUTPUT_DIR / 'bertopic_evolucao_temporal.png'}")
    plt.show()

    # V2 — heatmap mes x topico (% do mes)
    pct = serie.div(serie.sum(axis=1), axis=0).fillna(0) * 100
    fig, ax = plt.subplots(figsize=(max(9, serie.shape[1] * 0.5), max(4, len(serie) * 0.35)))
    sns.heatmap(pct.round(1), cmap="Blues", linewidths=0.4, ax=ax, cbar_kws={"label": "% do mes"})
    ax.set_title(f"% documentos por mes x topico — {corpus_id}")
    ax.set_xlabel("Topico"); ax.set_ylabel("Mes")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "bertopic_heatmap_temporal.png", dpi=150, bbox_inches="tight", facecolor="white")
    print(f"Salvo: {OUTPUT_DIR / 'bertopic_heatmap_temporal.png'}")
    plt.show()

    # V3 — painel por topico (small multiples)
    ncols = 3
    nrows = (serie.shape[1] + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 2.6 * nrows), sharex=True)
    axes = np.atleast_1d(axes).flatten()
    for i, tid in enumerate(serie.columns):
        axes[i].plot(serie.index, serie[tid], color="steelblue", linewidth=1.6)
        axes[i].fill_between(range(len(serie)), serie[tid], alpha=0.15, color="steelblue")
        axes[i].set_title(_label(tid), fontsize=9)
        axes[i].tick_params(axis="x", rotation=90, labelsize=7)
        axes[i].grid(alpha=0.25)
    for j in range(serie.shape[1], len(axes)):
        axes[j].axis("off")
    plt.suptitle(f"Evolucao por topico — {corpus_id}", y=1.005)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "bertopic_evolucao_por_topico.png", dpi=150, bbox_inches="tight", facecolor="white")
    print(f"Salvo: {OUTPUT_DIR / 'bertopic_evolucao_por_topico.png'}")
    plt.show()
else:
    print("Corpus sem date_column no params.yaml — figuras temporais puladas")